# 10 — Interpretabilidad

## Objetivo

Sustituir el SHAP sobre `hog_397` por explicaciones que un humano pueda leer:
coeficientes de la regresión logística sobre características de textura y un
**mapa espacial directo de HOG** (sin oclusión, sin re-segmentar).

## 0. Configuración

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA, V2, REP = ROOT / "data", ROOT / "data" / "v2", ROOT / "reports"
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 1. Coeficientes del modelo final

In [ ]:
coef = pd.read_csv(V2 / "lr_texture_coeficientes.csv")
print("Features seleccionadas:", len(coef))
coef.head(12).round(3)

In [ ]:
top = pd.concat([coef.head(8), coef.tail(8)]).sort_values("coefficient")
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#B2182B" if v > 0 else "#2166AC" for v in top["coefficient"]]
ax.barh(top["feature"], top["coefficient"], color=colors)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Coeficiente (espacio estandarizado)")
ax.set_title("LR textura: features que más empujan hacia TB")
plt.tight_layout()

Un coeficiente positivo aumenta el log-odds de TB; uno negativo lo reduce.
Como las features están estandarizadas, la magnitud es comparable dentro del
modelo. La interpretación clínica debe hacerse por familia (intensidad, GLCM,
LBP, Gabor), no por un índice aislado.

## 2. Importancia por familia

In [ ]:
from src.features import get_feature_groups
groups = get_feature_groups(pd.read_csv(DATA / "features_tbx11k_v2.csv", nrows=2))
fam = []
for name, cols in groups.items():
    sub = coef[coef["feature"].isin(cols)]
    if len(sub):
        fam.append({"familia": name, "n_seleccionadas": len(sub),
                    "sum_abs_coef": float(sub["coefficient"].abs().sum())})
pd.DataFrame(fam).sort_values("sum_abs_coef", ascending=False).round(3)

## 3. Mapa espacial HOG

In [ ]:
display(Image(filename=str(REP / "v2" / "hog_mapa_espacial.png")))

El mapa tiene 7×7 bloques (HOG con celdas 32×32 y bloques 2×2 sobre 256×256).
Cada bloque resume 36 features (4 celdas × 9 orientaciones). La intensidad del
mapa es la suma de |coeficiente| de las features HOG seleccionadas en ese
bloque. Es una atribución espacial directa, sin oclusión y sin re-segmentar.

## 4. Limitaciones

In [ ]:
manifest = json.loads((ROOT / "models" / "v2_lr_texture_manifest.json").read_text(encoding="utf-8"))
print("Features de entrada:", manifest["n_features_input"])
print("Features seleccionadas:", manifest["n_features_selected"])
print("Prevalencia:", round(manifest["prevalence"], 4))

- La atribución es **asociativa**, no causal.
- La máscara es un campo pulmonar con margen; parte de la textura es pared
  torácica.
- Los mapas HOG son gruesos (7×7); no localizan una lesión individual.
- La validación con cajas de lesión mide inclusión de la máscara, no
  localización diagnóstica del modelo.

## Conclusiones

La interpretabilidad ahora es legible: coeficientes por feature/familia y un
mapa espacial HOG reproducible. Es una ventaja real del enfoque clásico frente
a un SHAP sobre índices HOG sin significado.

**Fin del pipeline de modelado, evaluación e interpretabilidad.**